# 2. Data preparation and integration

We prepare the saved ElCom and BFS data and join them by BFS municipality number. The checks cover data types, missing values, duplicate keys, matches, exclusions and row counts before and after joining.

In [1]:
from pathlib import Path
import json
import pandas as pd

# Find the project directory.
working_directory = Path.cwd()

project_directory = (
    working_directory.parent
    if working_directory.name == "notebooks"
    else working_directory
)

raw_directory = project_directory / "data" / "raw"

print("Raw data directory", raw_directory)
print("Directory exists", raw_directory.is_dir())

Raw data directory data/raw
Directory exists True


In [2]:
elcom_files = list(
    raw_directory.glob("elcom_2026_h4_*.json")
)

# Exclude metadata files.
elcom_files = [
    file
    for file in elcom_files
    if not file.name.endswith("_metadata.json")
]

bfs_files = list(
    raw_directory.glob("bfs_municipalities_2026_*.csv")
)

# Select files automatically only when each source has one snapshot.
if len(elcom_files) != 1 or len(bfs_files) != 1:
    raise ValueError(
        "Expected exactly one ElCom response and one BFS CSV. "
        "Check the available snapshots."
    )

elcom_file = elcom_files[0]
bfs_file = bfs_files[0]

print("ElCom file", elcom_file.name)
print("BFS file", bfs_file.name)

ElCom file elcom_2026_h4_20261117T195442966207Z.json
BFS file bfs_municipalities_2026_20261117T195841888739Z.csv


In [3]:
# Load the original ElCom response.
with elcom_file.open(encoding="utf-8") as file:
    elcom_raw = json.load(file)

# Convert the nested response into table rows.
rows = [
    {
        field: entry["value"]
        for field, entry in record.items()
    }
    for record in elcom_raw["results"]["bindings"]
]

df_elcom = pd.DataFrame(
    rows,
    columns=elcom_raw["head"]["vars"]
).rename(columns={
    "gemeinde": "municipality_uri",
    "betreiber": "operator_uri",
    "gesamttarif": "total_tariff"
})

# Load the original BFS CSV.
df_bfs = pd.read_csv(
    bfs_file,
    encoding="utf-8-sig"
)

print("ElCom tariff rows", len(df_elcom))
print("BFS municipalities", len(df_bfs))

ElCom tariff rows 2268
BFS municipalities 2110


## 2.1 Prepare the tables for the join

ElCom embeds the municipality number in a URI, while BFS provides it in `BfsCode`. We extract the ElCom number and use the same column name and integer type in both tables. We convert the total tariff to a numeric type and label the BFS municipality types. This step retains all rows.

In [4]:
df_elcom_clean = df_elcom.copy()

# Extract the municipality number from the end of the URI.
df_elcom_clean["municipality_number"] = (
    df_elcom_clean["municipality_uri"]
    .str.extract(r"/municipality/(\d+)$", expand=False)
    .astype("Int64")
)

# Convert tariff values from text to numbers.
df_elcom_clean["total_tariff"] = pd.to_numeric(
    df_elcom_clean["total_tariff"],
    errors="raise"
)

df_elcom_clean.head()

,municipality_uri,operator_uri,total_tariff,municipality_number
0,https://ld.admin.ch/municipality/1,https://energy.ld.admin.ch/elcom/electricitypr...,24.136,1
1,https://ld.admin.ch/municipality/10,https://energy.ld.admin.ch/elcom/electricitypr...,24.136,10
2,https://ld.admin.ch/municipality/100,https://energy.ld.admin.ch/elcom/electricitypr...,24.136,100
3,https://ld.admin.ch/municipality/1001,https://energy.ld.admin.ch/elcom/electricitypr...,25.414,1001
4,https://ld.admin.ch/municipality/1002,https://energy.ld.admin.ch/elcom/electricitypr...,25.414,1002


In [5]:
# Select and rename the required fields.
df_bfs_clean = (
    df_bfs[
        ["BfsCode", "Name", "CantonId", "Canton", "STALAN2020"]
    ]
    .rename(
        columns={
            "BfsCode": "municipality_number",
            "Name": "municipality_name",
            "CantonId": "canton_id",
            "Canton": "canton",
            "STALAN2020": "municipality_type_code"
        }
    )
    .copy()
)

# Use the same key type as ElCom.
df_bfs_clean["municipality_number"] = (
    df_bfs_clean["municipality_number"].astype("Int64")
)

# Add readable municipality type labels.
type_labels = {
    1: "Urban",
    2: "Intermediate",
    3: "Rural"
}

df_bfs_clean["municipality_type"] = (
    df_bfs_clean["municipality_type_code"].map(type_labels)
)

df_bfs_clean.head()

,municipality_number,municipality_name,canton_id,canton,municipality_type_code,municipality_type
0,6158,Vionnaz,23,Valais / Wallis,2,Intermediate
1,3023,Speicher,15,Appenzell Ausserrhoden,2,Intermediate
2,6011,Zwischbergen,23,Valais / Wallis,3,Rural
3,2228,Villars-sur-Glâne,10,Fribourg / Freiburg,1,Urban
4,2230,Villarsel-sur-Marly,10,Fribourg / Freiburg,3,Rural


In [6]:
print("ElCom rows", len(df_elcom_clean))
print("BFS rows", len(df_bfs_clean))

print(
    "\nMissing ElCom municipality numbers",
    df_elcom_clean["municipality_number"].isna().sum()
)

print(
    "Missing BFS municipality type labels",
    df_bfs_clean["municipality_type"].isna().sum()
)

print(
    "\nElCom municipality number data type",
    df_elcom_clean["municipality_number"].dtype
)

print(
    "BFS municipality number data type",
    df_bfs_clean["municipality_number"].dtype
)

print(
    "Total tariff data type",
    df_elcom_clean["total_tariff"].dtype
)

print(
    "Missing total tariffs",
    df_elcom_clean["total_tariff"].isna().sum()
)

ElCom rows 2268
BFS rows 2110

Missing ElCom municipality numbers 0
Missing BFS municipality type labels 0

ElCom municipality number data type Int64
BFS municipality number data type Int64
Total tariff data type float64
Missing total tariffs 1


## 2.2 Join ElCom and BFS data

A left join adds the BFS municipality name, canton and municipality type to ElCom tariff rows. The relationship is many-to-one because several ElCom rows can belong to one BFS municipality. We flag unmatched rows and check that the join preserves the tariff row count.

In [7]:
df_join = df_elcom_clean.merge(
    df_bfs_clean,
    on="municipality_number",
    how="left",
    validate="many_to_one",
    indicator=True
)

df_join.head()

,municipality_uri,operator_uri,total_tariff,municipality_number,municipality_name,canton_id,canton,municipality_type_code,municipality_type,_merge
0,https://ld.admin.ch/municipality/1,https://energy.ld.admin.ch/elcom/electricitypr...,24.136,1,Aeugst am Albis,1.0,Zürich,3.0,Rural,both
1,https://ld.admin.ch/municipality/10,https://energy.ld.admin.ch/elcom/electricitypr...,24.136,10,Obfelden,1.0,Zürich,2.0,Intermediate,both
2,https://ld.admin.ch/municipality/100,https://energy.ld.admin.ch/elcom/electricitypr...,24.136,100,Stadel,1.0,Zürich,2.0,Intermediate,both
3,https://ld.admin.ch/municipality/1001,https://energy.ld.admin.ch/elcom/electricitypr...,25.414,1001,Doppleschwand,3.0,Luzern,3.0,Rural,both
4,https://ld.admin.ch/municipality/1002,https://energy.ld.admin.ch/elcom/electricitypr...,25.414,1002,Entlebuch,3.0,Luzern,3.0,Rural,both


In [8]:
print("ElCom rows before the join", len(df_elcom_clean))
print("Rows after the join", len(df_join))

print("\nTariff row matches")
print(df_join["_merge"].value_counts())

print(
    "\nDistinct matched municipalities",
    df_join.loc[
        df_join["_merge"] == "both",
        "municipality_number"
    ].nunique()
)

# Check that the left join preserves the number of tariff rows.
assert len(df_join) == len(df_elcom_clean), (
    "The join changed the row count."
)

ElCom rows before the join 2268
Rows after the join 2268

Tariff row matches
_merge
both          2251
left_only       17
right_only       0
Name: count, dtype: int64

Distinct matched municipalities 2105


In [9]:
unmatched = df_join.loc[
    df_join["_merge"] == "left_only",
    ["municipality_number", "municipality_uri", "operator_uri", "total_tariff"]
].copy()

with pd.option_context("display.max_colwidth", None):
    display(unmatched)

,municipality_number,municipality_uri,operator_uri,total_tariff
33,1057,https://ld.admin.ch/municipality/1057,https://energy.ld.admin.ch/elcom/electricityprice/operator/37,25.414
235,2016,https://ld.admin.ch/municipality/2016,https://energy.ld.admin.ch/elcom/electricityprice/operator/609,27.279
238,2027,https://ld.admin.ch/municipality/2027,https://energy.ld.admin.ch/elcom/electricityprice/operator/609,27.279
347,2278,https://ld.admin.ch/municipality/2278,https://energy.ld.admin.ch/elcom/electricityprice/operator/609,27.279
401,2456,https://ld.admin.ch/municipality/2456,https://energy.ld.admin.ch/elcom/electricityprice/operator/813,38.086
923,3932,https://ld.admin.ch/municipality/3932,https://energy.ld.admin.ch/elcom/electricityprice/operator/617,29.917
1040,4122,https://ld.admin.ch/municipality/4122,https://energy.ld.admin.ch/elcom/electricityprice/operator/616,26.820
1074,4179,https://ld.admin.ch/municipality/4179,https://energy.ld.admin.ch/elcom/electricityprice/operator/356,24.047
1314,5064,https://ld.admin.ch/municipality/5064,https://energy.ld.admin.ch/elcom/electricityprice/operator/712,29.952
1320,5078,https://ld.admin.ch/municipality/5078,https://energy.ld.admin.ch/elcom/electricityprice/operator/712,29.952


### Unmatched ElCom tariff rows

The initial join leaves 17 ElCom rows unmatched against the BFS register as of January 1, 2026. These rows have a total tariff. The join leaves their BFS municipality name, canton and type blank. We retain these rows and check the municipality numbers against official change notices.

In [10]:
bfs_without_elcom = df_bfs_clean.loc[
    ~df_bfs_clean["municipality_number"].isin(
        df_elcom_clean["municipality_number"]
    ),
    [
        "municipality_number",
        "municipality_name",
        "canton",
        "municipality_type"
    ]
].copy()

print("BFS municipalities without an ElCom tariff row:", len(bfs_without_elcom))

display(bfs_without_elcom)

BFS municipalities without an ElCom tariff row: 5


,municipality_number,municipality_name,canton,municipality_type
848,2822,Augst,Basel-Landschaft,Urban
2093,64,Nürensdorf,Zürich,Intermediate
2102,5395,Lema,Ticino,Intermediate
2105,6831,Moutier,Jura,Intermediate
2106,2056,Fétigny-Ménières,Fribourg / Freiburg,Rural


In [11]:
join_summary = pd.DataFrame({
    "Check": [
        "ElCom tariff rows before the join",
        "Tariff rows after the join",
        "Matched tariff rows",
        "Unmatched tariff rows",
        "BFS municipalities without an ElCom tariff row"
    ],
    "Count": [
        len(df_elcom_clean),
        len(df_join),
        (df_join["_merge"] == "both").sum(),
        (df_join["_merge"] == "left_only").sum(),
        len(bfs_without_elcom)
    ]
})

display(join_summary)

,Check,Count
0,ElCom tariff rows before the join,2268
1,Tariff rows after the join,2268
2,Matched tariff rows,2251
3,Unmatched tariff rows,17
4,BFS municipalities without an ElCom tariff row,5


## 2.3 Harmonize municipality numbers

We map ElCom municipality numbers to the BFS register as of January 1, 2026.

### Confirmed case of Moutier

ElCom uses municipality number 700 for Moutier. On January 1, 2026, Moutier moved from Bern to Jura and received BFS municipality number 6831. We join on 6831 and retain 700 as the original number. [Official BFS change notice](https://www.agvchapp.bfs.admin.ch/de/mutations/3999)

In [12]:
df_elcom_harmonized = df_elcom_clean.copy()

# Apply the officially confirmed change for Moutier.
number_mapping = {
    700: 6831
}

df_elcom_harmonized["municipality_number_2026"] = (
    df_elcom_harmonized["municipality_number"]
    .replace(number_mapping)
    .astype("Int64")
)

# Show changed municipality numbers.
display(
    df_elcom_harmonized.loc[
        df_elcom_harmonized["municipality_number"]
        != df_elcom_harmonized["municipality_number_2026"],
        ["municipality_number", "municipality_number_2026", "total_tariff"]
    ]
)

,municipality_number,municipality_number_2026,total_tariff
2085,700,6831,32.959


In [13]:
# Rename the BFS key for the join.
df_bfs_join = df_bfs_clean.rename(
    columns={"municipality_number": "municipality_number_2026"}
)

df_join_harmonized = df_elcom_harmonized.merge(
    df_bfs_join,
    on="municipality_number_2026",
    how="left",
    validate="many_to_one",
    indicator=True
)

# Check the match for Moutier.
display(
    df_join_harmonized.loc[
        df_join_harmonized["municipality_number"] == 700,
        [
            "municipality_number",
            "municipality_number_2026",
            "municipality_name",
            "canton",
            "municipality_type",
            "total_tariff",
            "_merge"
        ]
    ]
)

print("\nMatches after the first harmonization")
print(df_join_harmonized["_merge"].value_counts())

assert len(df_join_harmonized) == len(df_elcom_clean)

,municipality_number,municipality_number_2026,municipality_name,canton,municipality_type,total_tariff,_merge
2085,700,6831,Moutier,Jura,Intermediate,32.959,both



Matches after the first harmonization
_merge
both          2252
left_only       16
right_only       0
Name: count, dtype: int64


### Other officially confirmed municipality changes

The crosswalk maps former numbers to 2026 numbers and cites the official BFS change notice for each mapping. It preserves the original ElCom number and tariff. After a merger, rows from former municipality areas can share the same current municipality–operator pair. We inspect those rows before consolidating identical combinations.

In [14]:
# Columns contain the original number, the 2026 number,
# and the official BFS change notice number.

mappings = [
    (700, 6831, 3999),    # Moutier
    (1057, 1065, 3990),   # Honau -> Root
    (2016, 2056, 4000),   # Fétigny -> Fétigny-Ménières
    (2027, 2056, 4000),   # Ménières -> Fétigny-Ménières
    (2278, 2262, 4001),   # Ulmiz -> Gurmels
    (2456, 2465, 3985),   # Lüterswil-Gächliwil -> Buchegg
    (3932, 3901, 3989),   # Tschiertschen-Praden -> Chur
    (4122, 4095, 4003),   # Villnachern -> Brugg
    (4179, 4186, 3969),   # Ueken -> Herznach-Ueken
    (5064, 5073, 3995),   # Bodio -> Giornico
    (5078, 5079, 3996),   # Prato (Leventina) -> Quinto
    (5146, 5395, 3997),   # Astano -> Lema
    (5149, 5395, 3997),   # Bedigliora -> Lema
    (5181, 5395, 3997),   # Curio -> Lema
    (5200, 5395, 3997),   # Miglieglia -> Lema
    (5207, 5395, 3997),   # Novaggio -> Lema
    (6744, 6743, 3967)    # La Chaux-des-Breuleux -> Les Breuleux
]

df_number_mapping = pd.DataFrame(
    mappings,
    columns=[
        "original_municipality_number",
        "municipality_number_2026",
        "bfs_change_notice"
    ]
)

# Document sources in the mapping table.
df_number_mapping["source"] = (
    "https://www.agvchapp.bfs.admin.ch/de/mutations/"
    + df_number_mapping["bfs_change_notice"].astype(str)
)

# Check that each former number has exactly one mapping.
assert df_number_mapping["original_municipality_number"].is_unique

# Check that all mapped numbers occur in the selected BFS register.
assert df_number_mapping["municipality_number_2026"].isin(
    df_bfs_clean["municipality_number"]
).all()

display(df_number_mapping)

,original_municipality_number,municipality_number_2026,bfs_change_notice,source
0,700,6831,3999,https://www.agvchapp.bfs.admin.ch/de/mutations...
1,1057,1065,3990,https://www.agvchapp.bfs.admin.ch/de/mutations...
2,2016,2056,4000,https://www.agvchapp.bfs.admin.ch/de/mutations...
3,2027,2056,4000,https://www.agvchapp.bfs.admin.ch/de/mutations...
4,2278,2262,4001,https://www.agvchapp.bfs.admin.ch/de/mutations...
5,2456,2465,3985,https://www.agvchapp.bfs.admin.ch/de/mutations...
6,3932,3901,3989,https://www.agvchapp.bfs.admin.ch/de/mutations...
7,4122,4095,4003,https://www.agvchapp.bfs.admin.ch/de/mutations...
8,4179,4186,3969,https://www.agvchapp.bfs.admin.ch/de/mutations...
9,5064,5073,3995,https://www.agvchapp.bfs.admin.ch/de/mutations...


In [15]:
number_mapping = df_number_mapping.set_index(
    "original_municipality_number"
)["municipality_number_2026"].to_dict()

df_elcom_harmonized = df_elcom_clean.copy()

df_elcom_harmonized["municipality_number_2026"] = (
    df_elcom_harmonized["municipality_number"]
    .replace(number_mapping)
    .astype("Int64")
)

# Flag number changes for each tariff row.
df_elcom_harmonized["number_changed"] = (
    df_elcom_harmonized["municipality_number"]
    != df_elcom_harmonized["municipality_number_2026"]
)

df_bfs_join = df_bfs_clean.rename(
    columns={"municipality_number": "municipality_number_2026"}
)

df_join_harmonized = df_elcom_harmonized.merge(
    df_bfs_join,
    on="municipality_number_2026",
    how="left",
    validate="many_to_one",
    indicator=True
)

assert len(df_join_harmonized) == len(df_elcom_clean)

print("Tariff rows after harmonization",
      len(df_join_harmonized))

print("Tariff rows with a changed municipality number",
      df_join_harmonized["number_changed"].sum())

print("\nMatch status")
print(df_join_harmonized["_merge"].value_counts())

Tariff rows after harmonization 2268
Tariff rows with a changed municipality number 17

Match status
_merge
both          2268
left_only        0
right_only       0
Name: count, dtype: int64


In [16]:
repeated_pair_mask = df_join_harmonized.duplicated(
    subset=["municipality_number_2026", "operator_uri"],
    keep=False
)

repeated_pairs = df_join_harmonized.loc[
    repeated_pair_mask,
    [
        "municipality_number",
        "municipality_number_2026",
        "municipality_name",
        "operator_uri",
        "total_tariff"
    ]
].sort_values(
    ["municipality_number_2026", "operator_uri", "municipality_number"]
)

print(
    "Rows with repeated current "
    "municipality–operator combinations",
    len(repeated_pairs)
)

display(repeated_pairs)

Rows with repeated current municipality–operator combinations 23


,municipality_number,municipality_number_2026,municipality_name,operator_uri,total_tariff
33,1057,1065,Root,https://energy.ld.admin.ch/elcom/electricitypr...,25.414
42,1065,1065,Root,https://energy.ld.admin.ch/elcom/electricitypr...,25.153
235,2016,2056,Fétigny-Ménières,https://energy.ld.admin.ch/elcom/electricitypr...,27.279
238,2027,2056,Fétigny-Ménières,https://energy.ld.admin.ch/elcom/electricitypr...,27.279
337,2262,2262,Gurmels,https://energy.ld.admin.ch/elcom/electricitypr...,27.279
347,2278,2262,Gurmels,https://energy.ld.admin.ch/elcom/electricitypr...,27.279
401,2456,2465,Buchegg,https://energy.ld.admin.ch/elcom/electricitypr...,38.086
408,2465,2465,Buchegg,https://energy.ld.admin.ch/elcom/electricitypr...,38.086
917,3901,3901,Chur,https://energy.ld.admin.ch/elcom/electricitypr...,29.917
923,3932,3901,Chur,https://energy.ld.admin.ch/elcom/electricitypr...,29.917


## 2.4 Consolidate identical combinations after harmonization

After harmonization, 23 rows belong to repeated municipality–operator pairs. We retain each distinct tariff value once per current municipality and operator. This prevents identical values from gaining weight solely because they appeared in several former municipalities.

For example, the tariff values for the former Honau and Root differ, so both are retained. The harmonized provenance table records all original assignments.

In [17]:
# Select the fields needed for analysis.
# The harmonized join retains the full provenance.
df_distinct_tariffs = (
    df_join_harmonized[
        [
            "municipality_number_2026",
            "municipality_name",
            "canton_id",
            "canton",
            "municipality_type_code",
            "municipality_type",
            "operator_uri",
            "total_tariff",
            "_merge"
        ]
    ]
    .drop_duplicates(
        subset=[
            "municipality_number_2026",
            "operator_uri",
            "total_tariff"
        ]
    )
    .copy()
)

print("Rows before deduplication",
      len(df_join_harmonized))

print("Rows after deduplication",
      len(df_distinct_tariffs))

print("Duplicate rows consolidated",
      len(df_join_harmonized) - len(df_distinct_tariffs))

print("\nMissing total tariffs",
      df_distinct_tariffs["total_tariff"].isna().sum())

print("\nBFS match status")
print(df_distinct_tariffs["_merge"].value_counts())

Rows before deduplication 2268
Rows after deduplication 2256
Duplicate rows consolidated 12

Missing total tariffs 1

BFS match status
_merge
both          2256
left_only        0
right_only       0
Name: count, dtype: int64


In [18]:
display(
    df_distinct_tariffs.loc[
        df_distinct_tariffs["municipality_number_2026"] == 1065,
        [
            "municipality_number_2026",
            "municipality_name",
            "operator_uri",
            "total_tariff"
        ]
    ]
)

,municipality_number_2026,municipality_name,operator_uri,total_tariff
33,1065,Root,https://energy.ld.admin.ch/elcom/electricitypr...,25.414
42,1065,Root,https://energy.ld.admin.ch/elcom/electricitypr...,25.153


## 2.5 Handle the missing total tariff

Harmonization and deduplication leave 2,256 rows, all matched to a BFS municipality. We exclude one row with a missing total tariff and document it separately. The analysis uses observed tariff values only. We then check whether the affected municipality still has a valid tariff.

In [19]:
# Document the row with a missing total tariff.
df_excluded = df_distinct_tariffs.loc[
    df_distinct_tariffs["total_tariff"].isna()
].copy()

df_excluded["exclusion_reason"] = "Missing total tariff"

# Keep valid tariff rows for analysis.
df_analysis_tariffs = df_distinct_tariffs.loc[
    df_distinct_tariffs["total_tariff"].notna()
].copy()

print("Excluded tariff rows", len(df_excluded))
print("Tariff rows for analysis", len(df_analysis_tariffs))

display(df_excluded)

Excluded tariff rows 1
Tariff rows for analysis 2255


,municipality_number_2026,municipality_name,canton_id,canton,municipality_type_code,municipality_type,operator_uri,total_tariff,_merge,exclusion_reason
699,3218,Tübach,17,St. Gallen,1,Urban,https://energy.ld.admin.ch/elcom/electricitypr...,NaN,both,Missing total tariff


In [20]:
municipalities_with_tariffs = (
    df_analysis_tariffs["municipality_number_2026"].nunique()
)

df_municipalities_without_tariffs = df_bfs_clean.loc[
    ~df_bfs_clean["municipality_number"].isin(
        df_analysis_tariffs["municipality_number_2026"]
    ),
    [
        "municipality_number",
        "municipality_name",
        "canton",
        "municipality_type"
    ]
].copy()

print("Total BFS municipalities:", len(df_bfs_clean))
print("Municipalities with a valid tariff:", municipalities_with_tariffs)
print("Municipalities without a valid tariff:",
      len(df_municipalities_without_tariffs))

display(df_municipalities_without_tariffs)

# Check that both groups cover the full BFS register.
assert (
    municipalities_with_tariffs + len(df_municipalities_without_tariffs)
    == len(df_bfs_clean)
)

# Check that the split accounts for every row.
assert (
    len(df_analysis_tariffs) + len(df_excluded)
    == len(df_distinct_tariffs)
)

Total BFS municipalities: 2110
Municipalities with a valid tariff: 2108
Municipalities without a valid tariff: 2


,municipality_number,municipality_name,canton,municipality_type
848,2822,Augst,Basel-Landschaft,Urban
2093,64,Nürensdorf,Zürich,Intermediate


In [21]:
cleaning_summary = pd.DataFrame({
    "Step": [
        "ElCom tariff rows in the raw dataset",
        "Tariff rows with a changed municipality number",
        "Identical tariff combinations consolidated",
        "Tariff rows after deduplication",
        "Rows excluded due to a missing total tariff",
        "Remaining analysis rows",
        "Total BFS municipalities",
        "Municipalities with a valid tariff",
        "Municipalities without a valid tariff"
    ],
    "Count": [
        len(df_elcom_clean),
        df_join_harmonized["number_changed"].sum(),
        len(df_join_harmonized) - len(df_distinct_tariffs),
        len(df_distinct_tariffs),
        len(df_excluded),
        len(df_analysis_tariffs),
        len(df_bfs_clean),
        df_analysis_tariffs["municipality_number_2026"].nunique(),
        len(df_municipalities_without_tariffs)
    ]
})

display(cleaning_summary)

,Step,Count
0,ElCom tariff rows in the raw dataset,2268
1,Tariff rows with a changed municipality number,17
2,Identical tariff combinations consolidated,12
3,Tariff rows after deduplication,2256
4,Rows excluded due to a missing total tariff,1
5,Remaining analysis rows,2255
6,Total BFS municipalities,2110
7,Municipalities with a valid tariff,2108
8,Municipalities without a valid tariff,2


## 2.6 Data preparation results

All 2,268 raw tariff rows match the 2026 BFS register after harmonization, including the 17 rows unmatched by the initial join. Consolidating 12 identical municipality–operator–tariff combinations and excluding one row with a missing total yields 2,255 cleaned tariff rows across 2,108 municipalities.

Augst and Nürensdorf have no matching tariff in the saved snapshot. Both remain in the BFS reference table. The provenance table retains the full set of harmonized rows.

For the municipality comparison, we take the mean of distinct tariff values within each municipality and operator, then average the operator values within each municipality.

In [22]:
processed_directory = project_directory / "data" / "processed"
processed_directory.mkdir(parents=True, exist_ok=True)

tables = {
    "cleaned_tariffs.csv": df_analysis_tariffs,
    "bfs_municipalities_2026.csv": df_bfs_clean,
    "municipality_number_mapping.csv": df_number_mapping,
    "tariff_provenance.csv": df_join_harmonized,
    "excluded_tariffs.csv": df_excluded,
    "municipalities_without_tariffs.csv": df_municipalities_without_tariffs,
    "cleaning_summary.csv": cleaning_summary
}

for filename, table in tables.items():
    table.to_csv(
        processed_directory / filename,
        index=False,
        encoding="utf-8"
    )
    print("Saved", filename)

Saved cleaned_tariffs.csv
Saved bfs_municipalities_2026.csv
Saved municipality_number_mapping.csv
Saved tariff_provenance.csv
Saved excluded_tariffs.csv
Saved municipalities_without_tariffs.csv
Saved cleaning_summary.csv


In [23]:
# 1. Average tariff values within each municipality and operator.
df_operator_values = (
    df_analysis_tariffs
    .groupby(
        ["municipality_number_2026", "operator_uri"],
        as_index=False
    )["total_tariff"]
    .mean()
)

# 2. Average operator values within each municipality.
df_municipality_values = (
    df_operator_values
    .groupby("municipality_number_2026", as_index=False)
    .agg(municipality_tariff=("total_tariff", "mean"))
)

# 3. Add municipality name, canton and municipality type.
df_municipality_values = df_municipality_values.merge(
    df_bfs_clean.rename(
        columns={"municipality_number": "municipality_number_2026"}
    ),
    on="municipality_number_2026",
    how="left",
    validate="one_to_one"
)

# 4. Check the result.
assert len(df_municipality_values) == (
    df_analysis_tariffs["municipality_number_2026"].nunique()
)

assert df_municipality_values[
    ["municipality_tariff", "canton_id", "municipality_type"]
].notna().all().all()

print("Municipalities for analysis", len(df_municipality_values))

display(
    df_municipality_values[
        [
            "municipality_number_2026",
            "municipality_name",
            "canton",
            "municipality_type",
            "municipality_tariff"
        ]
    ].head()
)

Municipalities for analysis 2108


,municipality_number_2026,municipality_name,canton,municipality_type,municipality_tariff
0,1,Aeugst am Albis,Zürich,Rural,24.136
1,2,Affoltern am Albis,Zürich,Urban,24.136
2,3,Bonstetten,Zürich,Intermediate,24.136
3,4,Hausen am Albis,Zürich,Intermediate,24.136
4,5,Hedingen,Zürich,Urban,24.136


In [24]:
processed_directory = project_directory / "data" / "processed"
processed_directory.mkdir(parents=True, exist_ok=True)

df_operator_values.to_csv(
    processed_directory / "operator_tariff_values.csv",
    index=False,
    encoding="utf-8"
)

df_municipality_values.to_csv(
    processed_directory / "municipality_values.csv",
    index=False,
    encoding="utf-8"
)

print("Saved operator and municipality values.")

Saved operator and municipality values.
